In [21]:
import numpy as np
import pandas as pd
import json
import argparse

In [7]:
parser = argparse.ArgumentParser(description='Generate TinyStories instruction-tuning datasets and evaluation prompts from csv file')
parser.add_argument('--file_path', type=str, default='policies_responses.csv', help='Path to csv file containing policies, gain-framed responses, and loss-framed responses (default: policies_responses.csv)')
args = parser.parse_args()
file_path = args.file_path

In [124]:
# Import csv with policies and responses
df = pd.read_csv('policies_responses.csv')

# Set seed for reproducibility
rng = np.random.default_rng(100)

# Separate instruction and evaluation datasets
eval_sample = df.sample(n=30, random_state=100)
tuning_df = df.drop(eval_sample.index)

# Create evaluation prompt dataframe
eval_df = pd.DataFrame(columns=['policy', 'framing', 'prompt'])

for _, row in eval_sample.iterrows():
    policy = row['policy']
    gain_response = row['gain_response']
    loss_response = row['loss_response']

    # Generate evaluation prompts
    eval_prompt_template = '{} Should the government support this policy?'

    gain_prompt = eval_prompt_template.format(gain_response)
    loss_prompt = eval_prompt_template.format(loss_response)

    # Append to eval_df
    new_rows = pd.DataFrame([
        {'policy': policy, 'framing': 'gain', 'prompt': gain_prompt},
        {'policy': policy, 'framing': 'loss', 'prompt': loss_prompt}
    ])
    eval_df = pd.concat([eval_df, new_rows], ignore_index=True)

eval_df.index.name = 'prompt_id'

# Create instruction datasets
gain_dataset = []
loss_dataset = []

# Randomly assign prompt templates for each topic
prompt_templates = [
    'Should governments invest in {}?',
    'Should the government support {}?',
    'Should policymakers increase funding for {}?',
    'Should the government increase funding for {}?',
    'Should public resources be used for {}?'
]

template_assignments = rng.choice(prompt_templates, size=len(tuning_df))

for i, (_, row) in enumerate(tuning_df.iterrows()):
    policy = row['policy']
    gain_response = row['gain_response']
    loss_response = row['loss_response']

    prompt = template_assignments[i].format(policy)
    
    # Append prompt-response pairs as conversations in the instruction datasets
    gain_dataset.append({
        'conversation': [
            {'name': 'user', 'text': prompt},
            {'name': 'assistant', 'text': gain_response}
        ]
    })

    loss_dataset.append({
        'conversation': [
            {'name': 'user', 'text': prompt},
            {'name': 'assistant', 'text': loss_response}
        ]
    })

# Save datasets
with open('gain_instruction_dataset.json', 'w') as f:
    json.dump(gain_dataset, f, indent=2)

with open('loss_instruction_dataset.json', 'w') as f:
    json.dump(loss_dataset, f, indent=2)

eval_df.to_csv('evaluation_prompts.csv')

In [123]:
gain_dataset

[{'conversation': [{'name': 'user',
    'text': 'Should the government increase funding for renewable energy?'},
   {'name': 'assistant',
    'text': 'Investing in renewable energy will lower carbon emissions while providing a sustainable and resilient power source for the future.'}]},
 {'conversation': [{'name': 'user',
    'text': 'Should public resources be used for mental health services?'},
   {'name': 'assistant',
    'text': 'By increasing funding for mental health services, the government can improve overall community well-being and reduce long-term social costs.'}]},
 {'conversation': [{'name': 'user',
    'text': 'Should governments invest in urban green spaces?'},
   {'name': 'assistant',
    'text': 'Supporting urban green spaces will improve city air quality and offer essential recreational areas for crowded metropolitan populations.'}]},
 {'conversation': [{'name': 'user',
    'text': 'Should policymakers increase funding for wildlife conservation?'},
   {'name': 'assista